# 05 — Evaluación de DistilBERT sobre datos de test

Evalúa el pipeline `sentiment-analysis` de Hugging Face (**DistilBERT fine-tuned en SST-2**) sobre el conjunto de test de Sentiment140 y registra los resultados en MLflow.

In [1]:
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
from dotenv import load_dotenv
from transformers import pipeline, TextClassificationPipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report,
)

warnings.filterwarnings('ignore')
load_dotenv()

RANDOM_SEED            = int(os.getenv('RANDOM_SEED', 42))
MLFLOW_TRACKING_URI    = os.getenv('MLFLOW_TRACKING_URI', 'http://ec2-52-203-38-164.compute-1.amazonaws.com:5000')
MLFLOW_EXPERIMENT_NAME = os.getenv('MLFLOW_EXPERIMENT_NAME', 'sentiment140')
MEMBER                 = os.getenv('MEMBER', 'team')

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
np.random.seed(RANDOM_SEED)

DATA_PROCESSED = Path('../data/processed')

print('Setup complete')

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Setup complete


## Carga de datos de test

In [2]:
df_test    = pd.read_parquet(DATA_PROCESSED / 'test_clean.parquet')
y_test     = df_test['label'].values
X_test_raw = df_test['text'].fillna('').tolist()

print(f'Test size: {len(y_test):,}')

Test size: 240,000


## DistilBERT — HuggingFace Pipeline

Usando el patrón exacto de la skill NLP.

In [3]:
# Load pipeline (will download model ~260MB on first run)
sentiment_pipeline: TextClassificationPipeline = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    device=-1,       # -1 = CPU; set to 0 for GPU
    batch_size=64,
    truncation=True,
    max_length=512,
)
print('DistilBERT pipeline loaded.')

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

DistilBERT pipeline loaded.


In [4]:
# Quick sanity check (skill NLP pattern)
test_sentences = [
    'This is a wonderful movie!',
    'I absolutely hate this product.',
    'It is okay, nothing special.',
    'Amazing quality and fast delivery!',
]
print('Sentiment Analysis sanity check:')
for sentence in test_sentences:
    result = sentiment_pipeline(sentence)
    print(f'  Text: {sentence}')
    print(f'  Sentiment: {result[0]["label"]}, Score: {result[0]["score"]:.4f}')

Sentiment Analysis sanity check:
  Text: This is a wonderful movie!
  Sentiment: POSITIVE, Score: 0.9999
  Text: I absolutely hate this product.
  Sentiment: NEGATIVE, Score: 0.9997
  Text: It is okay, nothing special.
  Sentiment: NEGATIVE, Score: 0.8290
  Text: Amazing quality and fast delivery!
  Sentiment: POSITIVE, Score: 0.9999


In [5]:
# Evaluate on full test set
print(f'Running DistilBERT on {len(X_test_raw):,} test samples...')

t0 = time.time()
hf_outputs = sentiment_pipeline(X_test_raw, batch_size=64)
hf_latency = time.time() - t0

# DistilBERT labels: 'POSITIVE'/'NEGATIVE' → 1/0
hf_preds = np.array([1 if o['label'] == 'POSITIVE' else 0 for o in hf_outputs])

hf_metrics = {
    'accuracy':  accuracy_score(y_test, hf_preds),
    'precision': precision_score(y_test, hf_preds, average='macro'),
    'recall':    recall_score(y_test, hf_preds, average='macro'),
    'f1_macro':  f1_score(y_test, hf_preds, average='macro'),
    'latency_s': hf_latency,
}
print('\nDistilBERT (SST-2):')
for k, v in hf_metrics.items():
    print(f'  {k}: {v:.4f}')

print()
print(classification_report(y_test, hf_preds, target_names=['Negative', 'Positive']))

Running DistilBERT on 240,000 test samples...

DistilBERT (SST-2):
  accuracy: 0.7109
  precision: 0.7190
  recall: 0.7108
  f1_macro: 0.7081
  latency_s: 4914.9245

              precision    recall  f1-score   support

    Negative       0.68      0.81      0.74    120129
    Positive       0.76      0.61      0.68    119871

    accuracy                           0.71    240000
   macro avg       0.72      0.71      0.71    240000
weighted avg       0.72      0.71      0.71    240000



## Registro en MLflow

In [6]:
with mlflow.start_run(run_name='distilbert_sst2_test_eval') as run:
    mlflow.set_tag('member', MEMBER)
    mlflow.set_tag('stage', 'distilbert_eval')
    mlflow.set_tag('model', 'distilbert-base-uncased-finetuned-sst-2-english')
    mlflow.set_tag('type', 'reference_transformer')

    mlflow.log_params({
        'model':         'distilbert-base-uncased-finetuned-sst-2-english',
        'batch_size':    64,
        'device':        'cpu',
        'pretrained_on': 'SST-2',
        'fine_tuned':    False,
        'test_size':     len(y_test),
    })

    mlflow.log_metrics({
        'test_accuracy':       hf_metrics['accuracy'],
        'test_precision':      hf_metrics['precision'],
        'test_recall':         hf_metrics['recall'],
        'test_f1_macro':       hf_metrics['f1_macro'],
        'inference_latency_s': hf_metrics['latency_s'],
    })

    hf_run_id = run.info.run_id

# Save results JSON
results = {
    'model':         'distilbert-base-uncased-finetuned-sst-2-english',
    'pretrained_on': 'SST-2 (movie reviews)',
    'fine_tuned':    False,
    'test_size':     len(y_test),
    'metrics':       {k: round(v, 4) for k, v in hf_metrics.items()},
    'mlflow_run_id': hf_run_id,
}
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
(DATA_PROCESSED / 'distilbert_results.json').write_text(json.dumps(results, indent=2))

print(f'MLflow run_id: {hf_run_id}')
print(json.dumps(results, indent=2))

🏃 View run distilbert_sst2_test_eval at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11/runs/7a677bce776e47138ce7241d0f8351ce
🧪 View experiment at: http://ec2-52-203-38-164.compute-1.amazonaws.com:5000/#/experiments/11
MLflow run_id: 7a677bce776e47138ce7241d0f8351ce
{
  "model": "distilbert-base-uncased-finetuned-sst-2-english",
  "pretrained_on": "SST-2 (movie reviews)",
  "fine_tuned": false,
  "test_size": 240000,
  "metrics": {
    "accuracy": 0.7109,
    "precision": 0.719,
    "recall": 0.7108,
    "f1_macro": 0.7081,
    "latency_s": 4914.9245
  },
  "mlflow_run_id": "7a677bce776e47138ce7241d0f8351ce"
}
